# Pull data

In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3

## Functions

In [2]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name='dustin-payment-analysis'):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

## Constants

In [3]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii


## Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

## Read query

In [5]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('with tblMax as\n'
 '(\n'
 'select\n'
 '\tbigAccountId,\n'
 '\tmax(concat(MonthOnBooks, bigAccountid, bigRunDateKeyId)) as MaxUnique\n'
 'from riskdb.dbo.tblReportCOStaticPools_StaticPool\n'
 'where MonthOnBooks<=72\n'
 'group by bigAccountId\n'
 '),\n'
 '\n'
 '\n'
 '\n'
 'tblReportCOStaticPools_StaticPoolNew as\n'
 '(\n'
 'select\n'
 '\tconcat(MonthOnBooks, bigAccountid, bigRunDateKeyId) as uniqueC,\n'
 '\t*\n'
 'from riskdb.dbo.tblReportCOStaticPools_StaticPool\n'
 'where MonthOnBooks<=72\n'
 ')\n'
 '\n'
 '\n'
 'select \n'
 '\tCONCAT(tblAccount.bigAccountId, tblAccount.bigDebtorId, 1) as UniqueID,\n'
 '\ttbltempstaticpool.bigAccountId,\n'
 '\ttblAccount.bigDebtorId,\n'
 '\t1 as bitDebtor,\n'
 '\ttblAccount.dtmStampCreation,\n'
 '\ttbltempstaticpool.dtmFunded,\n'
 "\tcase when tbltempstaticpool.LoanStatusCNCombined='Default' then 1 else 0 "
 'end as bitDefault,\n'
 '\ttblReportCOStaticPools_StaticPoolNew.MonthOnBooks,\n'
 '\ttblReportCOStaticPools_StaticPoolNew.RunningNetLoss,\n'
 '\

## Write into df

In [6]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()
# info
print(f'Data contains {df.shape[0]} rows and {df.shape[1]} columns')

Data contains 196839 rows and 27 columns
Wall time: 41.5 s


In [7]:
# preview
df.head()

,UniqueID,bigAccountId,bigDebtorId,bitDebtor,dtmStampCreation,dtmFunded,bitDefault,MonthOnBooks,RunningNetLoss,AmtFinanced,...,fltDownCash,fltApprovedDownTotal,Payment,DTI,PTI,bitServiceContract,fltAdvance,strVehicleType,bitGap,DealerStampCreation
0,133767817566601,1337678,1756660,1,2013-10-01 10:24:32.527,2013-10-21,0,72,0.0,12152.00,...,1500.0,1500.0,314.19,0.222419,0.049405,0,0.940365,auto,0,2006-04-02 17:40:05.000
1,133774717567541,1337747,1756754,1,2013-10-01 11:01:41.130,2013-10-11,0,72,0.0,15831.73,...,500.0,500.0,364.53,0.510817,0.115263,0,1.048115,suv,1,2010-09-28 11:06:49.813
2,133778017567981,1337780,1756798,1,2013-10-01 11:25:23.807,2013-10-04,0,72,0.0,17335.26,...,1040.0,1040.0,394.88,0.396283,0.121472,0,1.121117,auto,0,2011-08-31 09:02:45.850
3,133780717568341,1337807,1756834,1,2013-10-01 11:41:51.333,2013-10-14,0,72,0.0,15422.00,...,500.0,500.0,413.68,0.505432,0.158861,0,1.094022,auto,0,2012-01-26 09:55:34.480
4,133781117568381,1337811,1756838,1,2013-10-01 11:44:35.337,2013-10-10,0,72,0.0,18570.70,...,1000.0,1000.0,450.11,0.482546,0.082613,1,1.146076,auto,1,2006-01-04 10:58:20.000


In [8]:
for col in df.columns:
    print(col)

UniqueID
bigAccountId
bigDebtorId
bitDebtor
dtmStampCreation
dtmFunded
bitDefault
MonthOnBooks
RunningNetLoss
AmtFinanced
intOpenBKType
intTerm
VehicleYear
bitNew
VehicleMake
Miles_Odometer
BookValue
fltDownCash
fltApprovedDownTotal
Payment
DTI
PTI
bitServiceContract
fltAdvance
strVehicleType
bitGap
DealerStampCreation


In [9]:
# get prop nan
df.isnull().mean()

UniqueID                0.000000
bigAccountId            0.000000
bigDebtorId             0.000000
bitDebtor               0.000000
dtmStampCreation        0.000000
dtmFunded               0.000000
bitDefault              0.000000
MonthOnBooks            0.000000
RunningNetLoss          0.000000
AmtFinanced             0.000000
intOpenBKType           0.580322
intTerm                 0.000000
VehicleYear             0.000000
bitNew                  0.000798
VehicleMake             0.000000
Miles_Odometer          0.000010
BookValue               0.000000
fltDownCash             0.000000
fltApprovedDownTotal    0.013412
Payment                 0.000000
DTI                     0.000122
PTI                     0.000000
bitServiceContract      0.000000
fltAdvance              0.000000
strVehicleType          0.000041
bitGap                  0.000000
DealerStampCreation     0.000000
dtype: float64

### Lower column names and add suffix

In [10]:
%%time

# list_cols = [
# ]

# # subset
# df = df[list_cols]

# lower columns
df.columns = [f'{col.lower()}__app' for col in df.columns]

# show
df

Wall time: 999 µs


,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,dtmstampcreation__app,dtmfunded__app,bitdefault__app,monthonbooks__app,runningnetloss__app,amtfinanced__app,...,fltdowncash__app,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app
0,133767817566601,1337678,1756660,1,2013-10-01 10:24:32.527,2013-10-21,0,72,0.0,12152.00,...,1500.0,1500.0,314.19,0.222419,0.049405,0,0.940365,auto,0,2006-04-02 17:40:05.000
1,133774717567541,1337747,1756754,1,2013-10-01 11:01:41.130,2013-10-11,0,72,0.0,15831.73,...,500.0,500.0,364.53,0.510817,0.115263,0,1.048115,suv,1,2010-09-28 11:06:49.813
2,133778017567981,1337780,1756798,1,2013-10-01 11:25:23.807,2013-10-04,0,72,0.0,17335.26,...,1040.0,1040.0,394.88,0.396283,0.121472,0,1.121117,auto,0,2011-08-31 09:02:45.850
3,133780717568341,1337807,1756834,1,2013-10-01 11:41:51.333,2013-10-14,0,72,0.0,15422.00,...,500.0,500.0,413.68,0.505432,0.158861,0,1.094022,auto,0,2012-01-26 09:55:34.480
4,133781117568381,1337811,1756838,1,2013-10-01 11:44:35.337,2013-10-10,0,72,0.0,18570.70,...,1000.0,1000.0,450.11,0.482546,0.082613,1,1.146076,auto,1,2006-01-04 10:58:20.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196834,481138661285821,4811386,6128583,0,2019-12-31 07:56:11.937,2020-02-18,0,44,0.0,22631.56,...,1000.0,1000.0,527.34,0.371485,0.053058,0,1.058707,suv,0,2018-11-21 11:01:46.307
196835,481150161287221,4811501,6128723,0,2019-12-31 09:10:49.117,2020-01-08,0,45,0.0,20024.75,...,500.0,500.0,454.07,0.300602,0.063171,1,1.234773,suv,0,2009-06-11 17:08:59.937
196836,481157161288091,4811571,6128810,0,2019-12-31 09:43:20.773,2020-01-31,0,45,0.0,15796.96,...,0.0,0.0,344.86,0.280712,0.100332,1,1.221195,auto,1,2019-08-09 13:53:46.860
196837,481178461290801,4811784,6129081,0,2019-12-31 11:14:46.693,2020-01-09,0,45,0.0,21666.29,...,1000.0,1000.0,443.03,0.271795,0.068680,0,1.106953,auto,1,2015-01-21 14:44:01.150


### Save

In [11]:
%%time

# save
str_filename = 'df_target_pd.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_csv(str_local_path, index=False)

Wall time: 9.77 s


## Upload to s3

In [12]:
# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'ad_hoc/target_pd/{str_filename}', 
    str_bucket_name=str_project,
)

## Clean-up

In [13]:
os.remove(str_local_path)